In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class HiDFLocalizationDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.real_dir = real_dir
        self.fake_dir = fake_dir
        self.transform = transform
        # Assuming fake images are named similarly to real images or mapped via a CSV
        self.fake_images = os.listdir(fake_dir)

    def __len__(self):
        return len(self.fake_images)

    def __getitem__(self, idx):
        fake_img_name = self.fake_images[idx]
        
        real_img_name = fake_img_name.split('_')[0] + ".jpg" 
        
        fake_path = os.path.join(self.fake_dir, fake_img_name)
        real_path = os.path.join(self.real_dir, real_img_name)

        # 1. Load Images
        fake_img = cv2.imread(fake_path)
        fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)
        
        real_img = cv2.imread(real_path)
        real_img = cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB)

        # 2. Generate the Ground Truth Mask (The "Difference Map")
        # Subtract real from fake, convert to grayscale
        diff = cv2.absdiff(fake_img, real_img)
        gray_diff = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
        
        # Apply threshold to create a binary mask (White = Fake, Black = Real)
        _, mask = cv2.threshold(gray_diff, 30, 255, cv2.THRESH_BINARY)
        mask = mask.astype(np.float32) / 255.0 # Normalize to 0-1

        # 3. Apply Transformations (Resize to patches, e.g., 224x224)
        if self.transform:
            augmented = self.transform(image=fake_img, mask=mask)
            fake_img = augmented['image']
            mask = augmented['mask']

        # Add channel dimension to mask [1, H, W]
        mask = mask.unsqueeze(0) 
        return fake_img, mask

# Standard Transformations for Swin Transformer (224x224)
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
import torch.nn as nn
import timm

class SwinUNet(nn.Module):
    def __init__(self, num_classes=1):
        super(SwinUNet, self).__init__()
        
        # ENCODER: Pre-trained Swin Transformer (captures global context)
        # Used 'features_only=True' to extract the hierarchical feature maps
        self.encoder = timm.create_model(
            'swin_tiny_patch4_window7_224', 
            pretrained=True, 
            features_only=True
        )
        
        # DECODER: U-Net style up-sampling
        # Swin-Tiny outputs channels: 96, 192, 384, 768
        self.upConv4 = nn.ConvTranspose2d(768, 384, kernel_size=2, stride=2)
        self.upConv3 = nn.ConvTranspose2d(384, 192, kernel_size=2, stride=2)
        self.upConv2 = nn.ConvTranspose2d(192, 96, kernel_size=2, stride=2)
        self.upConv1 = nn.ConvTranspose2d(96, 96, kernel_size=4, stride=4) # Final upsample to 224x224
        
        # Output Head: Binary Mask (Real vs Fake pixel)
        self.out_conv = nn.Conv2d(96, num_classes, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. Pass through Swin Encoder
        features = self.encoder(x)
        f1, f2, f3, f4 = features # Extract hierarchical patches
        
        # 2. Cascading Up-sampler (Simplified U-Net Decoder without skip-concat for brevity)
        d4 = self.upConv4(f4) 
        d3 = self.upConv3(d4 + f3) # Add skip connection
        d2 = self.upConv2(d3 + f2)
        d1 = self.upConv1(d2 + f1)
        
        # 3. Output Mask
        out = self.out_conv(d1)
        return self.sigmoid(out)

In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Hybrid Loss Function
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        bce_weight = 0.5
        bce = nn.BCELoss()(inputs, targets)
        
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()                            
        dice = 1 - (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)  
        
        return bce_weight * bce + (1 - bce_weight) * dice

# Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SwinUNet().to(device)
criterion = DiceBCELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

train_dataset = HiDFLocalizationDataset(real_dir='./data/real', fake_dir='./data/fake', transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Training Loop 
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    loop = tqdm(train_loader, leave=True)
    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
        
    print(f"Epoch {epoch+1} Average Loss: {running_loss/len(train_loader):.4f}")
    
    # Save checkpoint
    torch.save(model.state_dict(), f"swin_unet_epoch_{epoch+1}.pth")

In [ ]:
import matplotlib.pyplot as plt

def visualize_prediction(model, image_tensor, original_image):
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.unsqueeze(0).to(device) 
        prediction = model(image_tensor)
        
        pred_mask = prediction.squeeze().cpu().numpy()
        pred_mask = (pred_mask > 0.5).astype(np.uint8) # Threshold
        
    # Plotting
    fig, arr = plt.subplots(1, 3, figsize=(15, 5))
    
    arr[0].imshow(original_image)
    arr[0].set_title('Original Fake Image')
    
    arr[1].imshow(pred_mask, cmap='gray')
    arr[1].set_title('Predicted Forgery Mask')
    
    # Overlay Heatmap
    arr[2].imshow(original_image)
    arr[2].imshow(pred_mask, cmap='jet', alpha=0.5) # Alpha creates the overlay effect
    arr[2].set_title('Localization Heatmap')
    
    plt.show()
